# 6단계 · AI 보고서 생성

**목표:** 종합 결과를 **LLM이 해석**해 유망아이템 발굴 보고서를 자동 작성합니다.

## 구성
- 로컬 **Ollama**(OpenAI 호환) 서버의 **qwen3:8b** 모델 사용.
- 엔드포인트: `http://localhost:11434/v1`
- ⚠️ 이 노트북은 **Ollama 서버가 켜져 있어야** 동작합니다. (`ollama serve`, 모델 준비: `ollama pull qwen3:8b`)
- Qwen3는 기본 *thinking* 모드라, 보고서에 추론 텍스트가 섞이지 않게 `enable_thinking=False`로 끕니다.

> **1단계에서 확정한 시드를 사용합니다.**
> 데모 시드: **Mathematical finance** (itemId `33589680`, 카테고리 *인공지능*, 지표 *기술집약도*)

In [ ]:
# --- APOLLO API 호출 헬퍼 (모든 노트북 공통) ---
import requests, urllib3
import pandas as pd
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://apollo.kisti.re.kr/service-test"   # 테스트 서버

def call_api(method, path, params=None, payload=None):
    """APOLLO 호출 후 {meta, data} 본문을 반환. (테스트 서버라 verify=False)"""
    r = requests.request(method, BASE_URL + path, params=params, json=payload,
                         headers={"Content-Type": "application/json"},
                         verify=False, timeout=120)
    body = r.json()
    if not (isinstance(body, dict) and "data" in body):
        raise RuntimeError(f"API 오류 (HTTP {r.status_code}): {body}")
    return body

print("준비 완료 · BASE_URL =", BASE_URL)

In [ ]:
# 1단계에서 확정한 시드 (다른 데모 사례로 바꾸려면 값만 교체)
SEED_ID = 33589680          # Mathematical finance
SEED_NAME = "Mathematical finance"
CATEGORY = "인공지능"
DEGREE = 3                  # APOLLO degree=3  →  사용자 체감 '2차수'(이웃의 이웃)

### 종합 결과 준비 (3·5단계 요약)

In [ ]:
res = call_api("GET", f"/open/api/v1/items/{SEED_ID}/network/list",
               params={"degree": DEGREE})
df = pd.DataFrame(res["data"])
df["종합점수"] = df[["techIntensity", "demandEmergence", "supplyEmergence"]].mean(axis=1).round(2)
df = df.sort_values("종합점수", ascending=False).reset_index(drop=True)

rows = df.head(12)[["itemName", "category", "techIntensity",
                    "demandEmergence", "supplyEmergence", "종합점수"]].to_dict("records")
print("상위 아이템", len(rows), "건을 보고서 근거로 사용")

### LLM 호출 → 보고서

In [ ]:
# 로컬 LLM 직접 호출 (Ollama, OpenAI 호환 API) — 'openai' 패키지 불필요, requests만 사용
import json  # requests 는 위 셀에서 이미 import 됨

LLM_URL = "http://localhost:11434/v1/chat/completions"   # Ollama 엔드포인트
MODEL   = "qwen3:8b"                                      # `ollama list` 에 보이는 모델명

system = ("당신은 KISTI의 기술기획 분석가입니다. 주어진 유망아이템 지표 데이터를 근거로 "
          "교육생이 이해하기 쉬운 한국어 보고서를 작성합니다. 데이터에 없는 수치를 지어내지 말고, "
          "지표(기술집약도·수요/공급 부상도)의 의미를 풀어 해석하세요. 마크다운으로 간결하게.")
user = (f"시드 아이템: {SEED_NAME} (카테고리: {CATEGORY})\n\n"
        f"[연관 유망 아이템 상위 (JSON)]\n{json.dumps(rows, ensure_ascii=False, indent=2)}\n\n"
        "위 데이터로 다음 보고서를 작성하세요:\n"
        "## 1. 분석 개요\n## 2. 유망 아이템 TOP 3 해석\n## 3. 종합 시사점 및 제언")

payload = {
    "model": MODEL,
    "messages": [{"role": "system", "content": system},
                 {"role": "user", "content": user}],
    "temperature": 0.4,
    "max_tokens": 1800,
    "chat_template_kwargs": {"enable_thinking": False},   # Qwen3 thinking 끄기
}
resp = requests.post(LLM_URL, json=payload, timeout=120)
resp.raise_for_status()
report = resp.json()["choices"][0]["message"]["content"]
print(report)

**정리**
- 지표 수집(API) → 종합(코드) → 해석(LLM)까지, 유망아이템 발굴의 전 과정을 자동화했습니다.
- 보고서 초안을 사람이 검토·보완하면 완성됩니다.

수고하셨습니다! 🎉